# Evaluation and Standardized Comparison Table

In this notebook, we compare the four recommenders developed in the project:

- Non-Personalized
- Collaborative Filtering
- Content-Based
- Context-Aware

The final output is a standardized comparison table with the following columns:

Approach | RMSE | MAE | Precision@K | Recall@K | NDCG | Coverage | Diversity | Serendipity | Context

Contributors: Andrés Ramírez, Santiago Llorca

In [13]:
import pandas as pd
import numpy as np
from itertools import combinations
from sklearn.metrics.pairwise import cosine_similarity

## 1. Load Exported Predictions

In [14]:
np_df = pd.read_csv("predictions_non_personalized.csv")
cf_df = pd.read_csv("predictions_cf.csv")
content_df = pd.read_csv("predictions_content.csv")
context_df = pd.read_csv("predictions_context.csv")

print("Non-Personalized:", np_df.shape)
print("Collaborative Filtering:", cf_df.shape)
print("Content-Based:", content_df.shape)
print("Context-Aware:", context_df.shape)

Non-Personalized: (20168, 4)
Collaborative Filtering: (20168, 5)
Content-Based: (20168, 5)
Context-Aware: (20168, 6)


## 2. Merge All Prediction Files

In [15]:
eval_df = np_df.merge(cf_df, on=["userId", "movieId", "rating"], how="inner")
eval_df = eval_df.merge(content_df, on=["userId", "movieId", "rating"], how="inner")
eval_df = eval_df.merge(context_df, on=["userId", "movieId", "rating"], how="inner")

print("Merged evaluation dataframe:", eval_df.shape)
eval_df.head()

Merged evaluation dataframe: (20168, 11)


,userId,movieId,rating,score_non_personalized,score_cf,predicted_rating_cf,score_content,predicted_rating_content,context,score_context,predicted_rating_context
0,495,72998,5.0,3.662521,4.474270,4.474270,0.260112,4.783460,weekday,0.260112,4.783460
1,495,2762,4.5,3.866102,4.915041,4.915041,0.340262,4.660192,weekday,0.340262,4.660192
2,495,4993,0.5,4.093642,4.763956,4.763956,0.108660,4.825495,weekday,0.108660,4.825495
3,495,89492,5.0,3.632456,4.792751,4.792751,0.487505,4.680301,weekday,0.487505,4.680301
4,495,2028,4.5,4.130230,4.866406,4.866406,0.434230,4.641142,weekday,0.434230,4.641142


## 3. Load Original Data for Supporting Metrics

In [16]:
try:
    ratings = pd.read_csv("ml-latest-small/ratings.csv")
    movies = pd.read_csv("ml-latest-small/movies.csv")
except FileNotFoundError:
    ratings = pd.read_csv("ratings.csv")
    movies = pd.read_csv("movies.csv")

ratings = ratings.sort_values("timestamp").copy()
cutoff = int(len(ratings) * 0.8)

train = ratings.iloc[:cutoff].copy()
test = ratings.iloc[cutoff:].copy()

print("Train size:", train.shape)
print("Test size:", test.shape)

Train size: (80668, 4)
Test size: (20168, 4)


## 4. Build Item Similarity Matrix Again

This is needed for diversity and serendipity.

In [17]:
movies_features = movies.copy()
movies_features["genres"] = movies_features["genres"].str.replace("|", " ", regex=False)

genre_matrix = movies_features["genres"].str.get_dummies(sep=" ")
genre_matrix.index = movies_features["movieId"]

item_similarity = pd.DataFrame(
    cosine_similarity(genre_matrix),
    index=genre_matrix.index,
    columns=genre_matrix.index
)

item_similarity.head()

movieId,1,2,3,4,5,6,7,8,9,10,...,193565,193567,193571,193573,193579,193581,193583,193585,193587,193609
movieId,,,,,,,,,,,,,,,,,,,,,
1,1.000000,0.774597,0.316228,0.258199,0.447214,0.0,0.316228,0.632456,0.0,0.258199,...,0.447214,0.316228,0.316228,0.447214,0.0,0.670820,0.774597,0.00000,0.316228,0.447214
2,0.774597,1.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.816497,0.0,0.333333,...,0.000000,0.000000,0.000000,0.000000,0.0,0.288675,0.333333,0.00000,0.000000,0.000000
3,0.316228,0.000000,1.000000,0.816497,0.707107,0.0,1.000000,0.000000,0.0,0.000000,...,0.353553,0.000000,0.500000,0.000000,0.0,0.353553,0.408248,0.00000,0.000000,0.707107
4,0.258199,0.000000,0.816497,1.000000,0.577350,0.0,0.816497,0.000000,0.0,0.000000,...,0.288675,0.408248,0.816497,0.000000,0.0,0.288675,0.333333,0.57735,0.000000,0.577350
5,0.447214,0.000000,0.707107,0.577350,1.000000,0.0,0.707107,0.000000,0.0,0.000000,...,0.500000,0.000000,0.707107,0.000000,0.0,0.500000,0.577350,0.00000,0.000000,1.000000


## 5. Global Parameters

In [18]:
TOP_K = 10
RELEVANCE_THRESHOLD = 4.0
catalog_size = movies["movieId"].nunique()

## 6. Metric Functions

In [19]:
def rmse_metric(y_true, y_pred):
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)
    return np.sqrt(np.mean((y_true - y_pred) ** 2))


def mae_metric(y_true, y_pred):
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)
    return np.mean(np.abs(y_true - y_pred))


def precision_at_k(data, score_column, k=10, relevance_threshold=4.0):
    precisions = []

    for user in data["userId"].unique():
        user_data = data[data["userId"] == user].sort_values(score_column, ascending=False)
        top_k = user_data.head(k)

        if len(top_k) == 0:
            continue

        relevant = (top_k["rating"] >= relevance_threshold).sum()
        precisions.append(relevant / k)

    return np.mean(precisions) if precisions else np.nan


def recall_at_k(data, score_column, k=10, relevance_threshold=4.0):
    recalls = []

    for user in data["userId"].unique():
        user_data = data[data["userId"] == user]
        total_relevant = (user_data["rating"] >= relevance_threshold).sum()

        if total_relevant == 0:
            continue

        top_k = user_data.sort_values(score_column, ascending=False).head(k)
        relevant_in_top_k = (top_k["rating"] >= relevance_threshold).sum()

        recalls.append(relevant_in_top_k / total_relevant)

    return np.mean(recalls) if recalls else np.nan


def ndcg_at_k(data, score_column, k=10, relevance_threshold=4.0):
    ndcgs = []

    for user in data["userId"].unique():
        user_data = data[data["userId"] == user].sort_values(score_column, ascending=False).head(k)
        gains = (user_data["rating"] >= relevance_threshold).astype(int).values

        dcg = 0
        for i, rel in enumerate(gains, start=1):
            dcg += rel / np.log2(i + 1)

        ideal_gains = sorted(gains, reverse=True)
        idcg = 0
        for i, rel in enumerate(ideal_gains, start=1):
            idcg += rel / np.log2(i + 1)

        if idcg == 0:
            continue

        ndcgs.append(dcg / idcg)

    return np.mean(ndcgs) if ndcgs else np.nan


def coverage_at_k(data, score_column, catalog_size, k=10):
    recommended_items = set()

    for user in data["userId"].unique():
        user_data = data[data["userId"] == user].sort_values(score_column, ascending=False).head(k)
        recommended_items.update(user_data["movieId"].tolist())

    return len(recommended_items) / catalog_size if catalog_size > 0 else np.nan


def diversity_at_k(data, score_column, item_similarity, k=10):
    diversities = []

    for user in data["userId"].unique():
        top_k_items = (
            data[data["userId"] == user]
            .sort_values(score_column, ascending=False)
            .head(k)["movieId"]
            .tolist()
        )

        if len(top_k_items) < 2:
            continue

        pair_dissimilarities = []

        for i, j in combinations(top_k_items, 2):
            if i in item_similarity.index and j in item_similarity.columns:
                sim = item_similarity.loc[i, j]
                pair_dissimilarities.append(1 - sim)

        if pair_dissimilarities:
            diversities.append(np.mean(pair_dissimilarities))

    return np.mean(diversities) if diversities else np.nan


def serendipity_at_k(data, score_column, train, item_similarity, k=10, relevance_threshold=4.0):
    serendipities = []

    for user in data["userId"].unique():
        user_train_items = train[train["userId"] == user]["movieId"].unique().tolist()

        top_k = (
            data[data["userId"] == user]
            .sort_values(score_column, ascending=False)
            .head(k)
        )

        top_k = top_k[top_k["rating"] >= relevance_threshold]

        if top_k.empty:
            continue

        user_serendipity = []

        for rec_item in top_k["movieId"]:
            if rec_item not in item_similarity.index or len(user_train_items) == 0:
                continue

            seen_sims = []
            for seen_item in user_train_items:
                if seen_item in item_similarity.columns:
                    seen_sims.append(item_similarity.loc[rec_item, seen_item])

            if seen_sims:
                novelty = 1 - max(seen_sims)
                user_serendipity.append(novelty)

        if user_serendipity:
            serendipities.append(np.mean(user_serendipity))

    return np.mean(serendipities) if serendipities else np.nan

## 7. Compute Metrics for Each Approach

In [20]:
rmse_np = rmse_metric(eval_df["rating"], eval_df["score_non_personalized"])
mae_np = mae_metric(eval_df["rating"], eval_df["score_non_personalized"])

rmse_cf = rmse_metric(eval_df["rating"], eval_df["predicted_rating_cf"])
mae_cf = mae_metric(eval_df["rating"], eval_df["predicted_rating_cf"])

rmse_content = rmse_metric(eval_df["rating"], eval_df["predicted_rating_content"])
mae_content = mae_metric(eval_df["rating"], eval_df["predicted_rating_content"])

rmse_context = rmse_metric(eval_df["rating"], eval_df["predicted_rating_context"])
mae_context = mae_metric(eval_df["rating"], eval_df["predicted_rating_context"])

In [21]:
precision_np = precision_at_k(eval_df, "score_non_personalized", k=TOP_K, relevance_threshold=RELEVANCE_THRESHOLD)
recall_np = recall_at_k(eval_df, "score_non_personalized", k=TOP_K, relevance_threshold=RELEVANCE_THRESHOLD)
ndcg_np = ndcg_at_k(eval_df, "score_non_personalized", k=TOP_K, relevance_threshold=RELEVANCE_THRESHOLD)
coverage_np = coverage_at_k(eval_df, "score_non_personalized", catalog_size, k=TOP_K)
diversity_np = diversity_at_k(eval_df, "score_non_personalized", item_similarity, k=TOP_K)
serendipity_np = serendipity_at_k(eval_df, "score_non_personalized", train, item_similarity, k=TOP_K, relevance_threshold=RELEVANCE_THRESHOLD)

precision_cf = precision_at_k(eval_df, "score_cf", k=TOP_K, relevance_threshold=RELEVANCE_THRESHOLD)
recall_cf = recall_at_k(eval_df, "score_cf", k=TOP_K, relevance_threshold=RELEVANCE_THRESHOLD)
ndcg_cf = ndcg_at_k(eval_df, "score_cf", k=TOP_K, relevance_threshold=RELEVANCE_THRESHOLD)
coverage_cf = coverage_at_k(eval_df, "score_cf", catalog_size, k=TOP_K)
diversity_cf = diversity_at_k(eval_df, "score_cf", item_similarity, k=TOP_K)
serendipity_cf = serendipity_at_k(eval_df, "score_cf", train, item_similarity, k=TOP_K, relevance_threshold=RELEVANCE_THRESHOLD)

precision_content = precision_at_k(eval_df, "score_content", k=TOP_K, relevance_threshold=RELEVANCE_THRESHOLD)
recall_content = recall_at_k(eval_df, "score_content", k=TOP_K, relevance_threshold=RELEVANCE_THRESHOLD)
ndcg_content = ndcg_at_k(eval_df, "score_content", k=TOP_K, relevance_threshold=RELEVANCE_THRESHOLD)
coverage_content = coverage_at_k(eval_df, "score_content", catalog_size, k=TOP_K)
diversity_content = diversity_at_k(eval_df, "score_content", item_similarity, k=TOP_K)
serendipity_content = serendipity_at_k(eval_df, "score_content", train, item_similarity, k=TOP_K, relevance_threshold=RELEVANCE_THRESHOLD)

precision_context = precision_at_k(eval_df, "score_context", k=TOP_K, relevance_threshold=RELEVANCE_THRESHOLD)
recall_context = recall_at_k(eval_df, "score_context", k=TOP_K, relevance_threshold=RELEVANCE_THRESHOLD)
ndcg_context = ndcg_at_k(eval_df, "score_context", k=TOP_K, relevance_threshold=RELEVANCE_THRESHOLD)
coverage_context = coverage_at_k(eval_df, "score_context", catalog_size, k=TOP_K)
diversity_context = diversity_at_k(eval_df, "score_context", item_similarity, k=TOP_K)
serendipity_context = serendipity_at_k(eval_df, "score_context", train, item_similarity, k=TOP_K, relevance_threshold=RELEVANCE_THRESHOLD)

## 8. Final Standardized Comparison Table

In [22]:
results = pd.DataFrame({
    "Approach": ["Non-Personalized", "Collaborative Filtering", "Content-Based", "Context-Aware"],
    "RMSE": [rmse_np, rmse_cf, rmse_content, rmse_context],
    "MAE": [mae_np, mae_cf, mae_content, mae_context],
    "Precision@K": [precision_np, precision_cf, precision_content, precision_context],
    "Recall@K": [recall_np, recall_cf, recall_content, recall_context],
    "NDCG": [ndcg_np, ndcg_cf, ndcg_content, ndcg_context],
    "Coverage": [coverage_np, coverage_cf, coverage_content, coverage_context],
    "Diversity": [diversity_np, diversity_cf, diversity_content, diversity_context],
    "Serendipity": [serendipity_np, serendipity_cf, serendipity_content, serendipity_context],
    "Context": ["No", "No", "No", "Weekday/Weekend"]
})

results.round(4)

,Approach,RMSE,MAE,Precision@K,Recall@K,NDCG,Coverage,Diversity,Serendipity,Context
0,Non-Personalized,1.0268,0.8010,0.7000,0.2420,0.8894,0.0253,0.6601,0.0992,No
1,Collaborative Filtering,1.0714,0.8444,0.6966,0.2497,0.9141,0.0520,0.6317,0.1079,No
2,Content-Based,1.1072,0.8661,0.6836,0.2450,0.8935,0.0554,0.5704,0.0660,No
3,Context-Aware,1.1027,0.8622,0.6888,0.2470,0.8949,0.0543,0.5865,0.0761,Weekday/Weekend


## 9 A/B Testing

To simulate an online comparison, we would test the two strongest candidate models:

- Variant A: Non-Personalized Recommender
- Variant B: Collaborative Filtering Recommender

Users would be randomly assigned to one of the two variants.

The main KPI would be Hit Rate@10, defined as whether at least one relevant item (rating >= 4) appears in the user’s top-10 recommendations.

Secondary KPIs could include Precision@10 and NDCG@10.

The winning model would be the one that achieves the best balance between user relevance and recommendation quality.

In [23]:
np.random.seed(1)

ab_users = pd.DataFrame({
    "userId": eval_df["userId"].unique(),
    "group": np.random.choice(["A", "B"], size=eval_df["userId"].nunique())
})

ab_df = eval_df.merge(ab_users, on="userId", how="left")
ab_df.head()

,userId,movieId,rating,score_non_personalized,score_cf,predicted_rating_cf,score_content,predicted_rating_content,context,score_context,predicted_rating_context,group
0,495,72998,5.0,3.662521,4.474270,4.474270,0.260112,4.783460,weekday,0.260112,4.783460,B
1,495,2762,4.5,3.866102,4.915041,4.915041,0.340262,4.660192,weekday,0.340262,4.660192,B
2,495,4993,0.5,4.093642,4.763956,4.763956,0.108660,4.825495,weekday,0.108660,4.825495,B
3,495,89492,5.0,3.632456,4.792751,4.792751,0.487505,4.680301,weekday,0.487505,4.680301,B
4,495,2028,4.5,4.130230,4.866406,4.866406,0.434230,4.641142,weekday,0.434230,4.641142,B


In [24]:
def hit_rate_at_k(data, score_column, k=10, relevance_threshold=4.0):
    hits = []

    for user in data["userId"].unique():
        top_k = (
            data[data["userId"] == user]
            .sort_values(score_column, ascending=False)
            .head(k)
        )
        hits.append(int((top_k["rating"] >= relevance_threshold).any()))

    return np.mean(hits)

In [25]:
group_a = ab_df[ab_df["group"] == "A"]
group_b = ab_df[ab_df["group"] == "B"]

hit_rate_a = hit_rate_at_k(group_a, "score_non_personalized", k=10, relevance_threshold=4.0)
hit_rate_b = hit_rate_at_k(group_b, "score_cf", k=10, relevance_threshold=4.0)

ab_results = pd.DataFrame({
    "Variant": ["A", "B"],
    "Model": ["Non-Personalized", "Collaborative Filtering"],
    "HitRate@10": [hit_rate_a, hit_rate_b]
})

ab_results

,Variant,Model,HitRate@10
0,A,Non-Personalized,0.980392
1,B,Collaborative Filtering,0.969231


## Final Summary

This notebook presents the evaluation phase of the recommender systems project.

We evaluated the four recommenders using multiple metric families: prediction accuracy (RMSE, MAE), ranking quality (Precision@K, Recall@K, NDCG), and beyond-accuracy metrics (Coverage, Diversity, Serendipity).

The evaluation was conducted offline using a temporal train/test split. This split was first created inside each recommender notebook and then reproduced here for consistency. Using a temporal split is more realistic than a random split because models must always predict future interactions from past data.

We also included a simulated A/B test design to compare two candidate recommenders before deployment. Although this is not a real online experiment, it is a useful offline approximation for model selection before production testing. :contentReference[oaicite:3]{index=3}

Overall, this notebook provides a unified framework to compare the four recommendation approaches and serves as the basis for the final interpretation in the report.